In [ ]:
!pip install yadisk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 3.2 MB/s eta 0:00:00


In [ ]:
import json
import yadisk
import os

import cv2
import numpy as np

from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
def encode(frames):
    embedding=[1,2,3]

    return embedding

In [ ]:
def extract_frames(cap, opened_file, window_size=11, overlap=5, new_fps=3):
    embeddings = dict() # {первый кадр: эмбеддинг}
    window_frames = []
    wfi = []
    i = 0

    frame_num = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) # frames per second
    fpw = int(fps*window_size) # frames per window
    new_fpw = int(new_fps*window_size)
    new_fpo = int(new_fps*overlap) # frames per overlap
    step = int(fps/new_fps)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if i % step == 0 and len(window_frames) < new_fpw:
            window_frames.append(frame)
            wfi.append(i)
        if len(window_frames) >= new_fpw:
            emb = encode(window_frames)
            embeddings[wfi[0]] = emb
            window_frames = window_frames[new_fpo:]
            wfi = wfi[new_fpo:]
        i += 1
    del window_frames
    cap.release() # закрываем исходный файл
    os.remove(opened_file) # удаляем исходный файл
    return embeddings

In [ ]:
def read_data(data_items):
        embeddings_train = []
        texts_train = []
        embeddings_eval = []
        texts_eval = []
        embeddings_test = []
        texts_test = []

        # Сортируем data_items по video
        data_items = sorted(data_items, key=lambda x: x['video'])
        opened_file = ''
        last_cap_embs = None

        for data_item in tqdm(data_items):
            if data_item['video'] != opened_file:
                # скачиваем новый файлик
                opened_file = data_item['video']
                TOKEN = os.getenv('DEBUG_TOKEN')
                client = yadisk.Client(token=TOKEN)
                path_on_disk = 'disk:/SLR Project/' + opened_file
                client.download(path_on_disk, opened_file) # скачиваем файл
                cap = cv2.VideoCapture(opened_file)
                last_cap_embs = extract_frames(cap, opened_file)
            emb = last_cap_embs[data_item['frame_ids'][0]]
            if data_item['split'] == 'train':
                embeddings_train.append(emb)
                texts_train.append(data_item['text'])
            if data_item['split'] == 'eval':
                embeddings_eval.append(emb)
                texts_eval.append(data_item['text'])
            if data_item['split'] == 'test':
                embeddings_test.append(emb)
                texts_test.append(data_item['text'])

        del last_cap_embs
        del frame_list

        return (embeddings_train, texts_train,
                embeddings_eval, texts_eval,
                embeddings_test, texts_test)

In [ ]:
class SLDataset(Dataset):
    def __init__(self, embeddings, texts):
        self.embeddings = embeddings
        self.texts = texts

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        sample = self.embeddings[idx]
        label = self.texts[idx]
        return {'emb': sample, 'text':label}

In [ ]:
with open('test.json', 'r', encoding='utf-8') as f:
    test_samples = json.load(f)

In [ ]:
embeddings_train, texts_train, embeddings_eval, texts_eval, embeddings_test, texts_test = read_data(test_samples)
test_dataset = SLDataset(embeddings_test, texts_test)
del embeddings_test
del texts_test

  0%|          | 0/2633 [00:00<?, ?it/s]

[0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96]
[45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96]
[45, 48, 51, 54, 57, 60, 63, 66, 69, 72, 75, 78, 81, 84, 87, 90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141]
[90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141]
[90, 93, 96, 99, 102, 105, 108, 111, 114, 117, 120, 123, 126, 129, 132, 135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186]
[135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186]
[135, 138, 141, 144, 147, 150, 153, 156, 159, 162, 165, 168, 171, 174, 177, 180, 183, 186, 189, 192, 195, 198, 201, 204, 207, 210, 213, 216, 219, 222, 225, 228, 231]
[180, 183, 186, 189, 192, 195, 198, 201, 204, 207, 210, 213, 216, 219, 222, 225, 228, 231]
[180, 183, 186, 189, 192, 195, 198, 201, 204, 207, 210

  0%|          | 0/2633 [00:15<?, ?it/s]


KeyboardInterrupt: 